In [1]:
# IMPORTS

from classes.single_encoder import SingleEncoder
from classes.dual_encoder_single import DualEncoderAsSingle
from helpers.generate_embeddings import generate_embeddings
from helpers.load_embeddings import load_embeddings
from helpers.build_prompt import build_prompt
from helpers.retrieve_top_k import retrieve_top_k
from helpers.load_dual_encoder import load_dual_encoder_model
from helpers.load_single_encoder import load_single_encoder_model
from helpers.retrieve_weight_samples import retrieve_weighted_samples
from transformers import CanineModel, CanineTokenizer
import pandas as pd
import torch
import faiss
import numpy as np
import pickle
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0114 18:53:29.946000 12656 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# SETUP
MODEL_PATH = "../output/models/"
EMBEDDINGS_PATH = "../output/embeddings/"
CHAT_DATA_PATH = "../data/processed/"

MODEL_NAME = "author_contrastive"
# MODEL_NAME = "style_discovery_loss"
CHAT_DATA = "private_full"


MODEL_PATH = MODEL_PATH + MODEL_NAME
EMBEDDINGS_PATH = EMBEDDINGS_PATH + MODEL_NAME
CHAT_DATA_PATH = CHAT_DATA_PATH + CHAT_DATA + ".csv"

#=====================================
# LOAD MODEL
# Base CANINE
tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
encoder = CanineModel.from_pretrained("google/canine-s")
model = SingleEncoder()
model.encoder = encoder  # replace the encoder
model.to(device)
model.eval()

# Author Contrastive Loss
# model, tokenizer, device = load_single_encoder_model(MODEL_PATH)

# Style Discovery Loss
# model, tokenizer, device = load_dual_encoder_model(MODEL_PATH)
# model = DualEncoderAsSingle(model)

SingleEncoder(
  (encoder): CanineModel(
    (char_embeddings): CanineEmbeddings(
      (HashBucketCodepointEmbedder_0): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_1): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_2): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_3): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_4): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_5): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_6): Embedding(16384, 96)
      (HashBucketCodepointEmbedder_7): Embedding(16384, 96)
      (char_position_embeddings): Embedding(16384, 768)
      (token_type_embeddings): Embedding(16, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (initial_char_encoder): CanineEncoder(
      (layer): ModuleList(
        (0): CanineLayer(
          (attention): CanineAttention(
            (self): CanineSelfAttention(
              (query): Linear

In [8]:
# LOAD CHAT DATA AND GENERATE EMBEDDINGS
df = pd.read_csv(CHAT_DATA_PATH)
df = df.dropna(subset=["Content"])
df["Content"] = df["Content"].astype(str)

print("Messages:", len(df))
print("Authors:", df["Author"].nunique())

texts = df["Content"].tolist()

embeddings = generate_embeddings(
    texts=texts,
    model=model,
    tokenizer=tokenizer,
    batch_size=8,
    device=device,
)

print("Embedding shape:", embeddings.shape)

records = [
    {
        "author": df.iloc[i]["Author"],
        "content": df.iloc[i]["Content"]
    }
    for i in range(len(df))
]

# Save to FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity
index.add(embeddings.numpy())

print("Vectors in index:", index.ntotal)

os.makedirs(EMBEDDINGS_PATH, exist_ok=True)

faiss.write_index(index, os.path.join(EMBEDDINGS_PATH, "style_index.faiss"))
with open(os.path.join(EMBEDDINGS_PATH, "style_metadata.pkl"), "wb") as f:
    pickle.dump(records, f)
torch.save(embeddings, os.path.join(EMBEDDINGS_PATH, "style_embeddings.pt"))

Messages: 40725
Authors: 2


100%|██████████| 5091/5091 [02:05<00:00, 40.51it/s]


Embedding shape: torch.Size([40725, 128])
Vectors in index: 40725


In [3]:
# load embeddings directly if already generated
index, records, embeddings = load_embeddings(EMBEDDINGS_PATH)

In [18]:
# Example query
query_text = "what's up bruh"

# Retrieve weighted samples
retrieved_results = retrieve_weighted_samples(
    query_text=query_text,
    model=model,
    tokenizer=tokenizer,
    index=index,
    records=records,
    device=device,
    k=10,
    temperature=0.03  # Adjust for more/less diversity
)

# Print results
for i, result in enumerate(retrieved_results):
    print(f"\n--- Result {i+1} ---")
    print(f"Author: {result['author']}")
    print(f"Content: {result['content']}")
    print(f"Similarity Rank: {result['rank_by_similarity']}")
    print(f"Probability Rank: {result['rank_by_probability']}")


--- Result 1 ---
Author: e1688f3c
Content: creaate a new channel
Similarity Rank: 1
Probability Rank: 1

--- Result 2 ---
Author: fa09cb53
Content: synthesizers included in fl is also a choice but making a sound is long and hard 😩
Similarity Rank: 5
Probability Rank: 5

--- Result 3 ---
Author: e1688f3c
Content: khi nguoi gia vo dang nghi
Similarity Rank: 7
Probability Rank: 7

--- Result 4 ---
Author: fa09cb53
Content: the other mfs are straight af
Similarity Rank: 10
Probability Rank: 10

--- Result 5 ---
Author: fa09cb53
Content: theres only one synth doing the major shit
Similarity Rank: 15
Probability Rank: 15

--- Result 6 ---
Author: e1688f3c
Content: executing baiden no memory wipe
Similarity Rank: 35
Probability Rank: 35

--- Result 7 ---
Author: e1688f3c
Content: there aren't any computer courses
Similarity Rank: 43
Probability Rank: 43

--- Result 8 ---
Author: e1688f3c
Content: testing out all the discord server templates 👍
Similarity Rank: 59
Probability Rank: 59

--- Res